# Part 3: Visualising Convolutional Networks

__Before starting, run the setup cell below. If `bettercnn.weights` is not already present, Colab will ask you to upload the weights saved in Part 1.__

In [ ]:
# Download the supplied examples when the notebook is opened directly in Colab.
from pathlib import Path
from urllib.request import urlretrieve

base_url = "https://raw.githubusercontent.com/ecs-vlc/AICE3002_6001/main/labs/06-CNNs"
for filename in [f"{digit}.PNG" for digit in range(10)]:
    if not Path(filename).exists():
        urlretrieve(f"{base_url}/{filename}", filename)

# If you did not create the weights in Part 1, Colab will offer an upload button.
if not Path("bettercnn.weights").exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Place bettercnn.weights in this directory before continuing.")

## Visualising the first layers filters and responses

In our previous `BetterCNN` convolutional network, the first layer was a Convolutional layer. Because this convolutional layer is applied directly to the greylevel input MNIST images the filters that are learned can themselves just be considered to be small (5x5 in this case) greylevel images. 

We'll start by doing a few imports and then loading our pre-trained model. Once again, please copy-paste the forward method from the first workbook:

In [ ]:
%matplotlib inline

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import nn

# Model Definition
class BetterCNN(nn.Module):
    def __init__(self):
        super(BetterCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, (5, 5), padding=0)
        self.conv2 = nn.Conv2d(32, 15, (3, 3), padding=0)
        self.fc1 = nn.Linear(15 * 5**2, 128)
        self.fc2 = nn.Linear(128, 50)
        self.fc3 = nn.Linear(50, 10)
    
    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

# build the model and load state
model = BetterCNN()
model.load_state_dict(torch.load('bettercnn.weights', map_location='cpu', weights_only=True))
model.eval()

We can extract the weights of the first layer filters directly from the trained network and visualise them using `matplotlib` like this:

In [ ]:
weights = model.conv1.weight.detach().cpu()

# plot the first-layer filters
for i in range(30):
    plt.subplot(5, 6, i + 1)
    plt.imshow(weights[i, 0], cmap=plt.get_cmap('gray'))
    plt.axis('off')
plt.show()

Note that `model.conv1.weight` is the tensor holding the weights. Calling `cpu()` ensures data is moved over from the GPU if necessary.

__Answer the following question (enter the answer in the box below):__

__1.__ What sort of features do the filters resemble? How does this relate to your knowledge of the training data?

YOUR ANSWER HERE

## Visualising feature maps

If we forward propagate an input through the network we can also visualise the response maps generated by the filters. The advantage of this kind of visualisation is that we can compute it at any layer, not just the first one. In order to do this in PyTorch, we can propagate the given input through the network to the required point and use a `hook` to intercept the feature maps as they are created. The following code shows how this can be achieved to generate the response maps of the second convolutional layer of our network:

In [ ]:
from PIL import Image
import torchvision

transform = torchvision.transforms.ToTensor()
im = transform(Image.open("1.PNG")).unsqueeze(0)

def hook_function(module, inputs, output):
    feature_maps = output.detach()[0]
    for i in range(feature_maps.shape[0]):
        plt.subplot(5, int(1 + feature_maps.shape[0] / 5), i + 1)
        plt.imshow(feature_maps[i], cmap=plt.get_cmap('gray'))
        plt.axis('off')

hook = model.conv2.register_forward_hook(hook_function)
with torch.inference_mode():
    model(im)
hook.remove()
plt.show()

__Use the following code block to visualise the feature maps of the first convolutional layer__:

A final way of visualising what the filters (at any depth) are learning is to find the input image that maximises the response of the filter. We can do this by starting with a random image and using gradient ascent to optimise the image to maximise the chosen filter (see http://www.iro.umontreal.ca/~lisa/publications2/index.php/publications/show/247 and https://distill.pub/2017/feature-visualization/ for more info on this approach). The following code snippet shows how this can be achieved:

In [ ]:
def visualise_maximum_activation(model, target, num=10, alpha=1.0):
    model.eval()
    model.requires_grad_(False)
    plt.figure(figsize=(10, 4))

    for selected in range(num):
        input_img = torch.randn(1, 1, 28, 28, requires_grad=True)
        selected_output = None

        def hook_function(module, inputs, output):
            nonlocal selected_output
            selected_output = output[0, selected]

        hook = target.register_forward_hook(hook_function)

        for _ in range(30):
            if input_img.grad is not None:
                input_img.grad.zero_()

            model(input_img)
            selected_output.mean().backward()

            with torch.no_grad():
                normaliser = input_img.grad.std() + 1e-5
                input_img += alpha * input_img.grad / normaliser

        hook.remove()

        plt.subplot(2, (num + 1) // 2, selected + 1)
        plt.imshow(input_img.detach()[0, 0], cmap=plt.get_cmap('gray'))
        plt.axis('off')

    plt.show()

visualise_maximum_activation(model, model.fc3)